In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
#modeling 
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

In [29]:
df = pd.read_csv("data/health_lifestyle_dataset.csv")
df.head()

,id,age,gender,bmi,daily_steps,sleep_hours,water_intake_l,calories_consumed,smoker,alcohol,resting_hr,systolic_bp,diastolic_bp,cholesterol,family_history,disease_risk
0,1,56,Male,20.5,4198,3.9,3.4,1602,0,0,97,161,111,240,0,0
1,2,69,Female,33.3,14359,9.0,4.7,2346,0,1,68,116,65,207,0,0
2,3,46,Male,31.6,1817,6.6,4.2,1643,0,1,90,123,99,296,0,0
3,4,32,Female,38.2,15772,3.6,2.0,2460,0,0,71,165,95,175,0,0
4,5,60,Female,33.6,6037,3.8,4.0,3756,0,1,98,139,61,294,0,0


In [30]:
data = (df.sample(n=10000, random_state=42).reset_index(drop=True).drop(columns=['id']))

In [31]:
data.shape

(10000, 15)

In [32]:
X = data.drop(columns=['disease_risk'],axis=1)

In [33]:
X.head(1)

,age,gender,bmi,daily_steps,sleep_hours,water_intake_l,calories_consumed,smoker,alcohol,resting_hr,systolic_bp,diastolic_bp,cholesterol,family_history
0,67,Female,25.2,9979,8.8,1.8,2792,0,1,73,110,95,171,0


In [34]:
y = data['disease_risk']

In [35]:
y.head()

0    0
1    1
2    1
3    0
4    0
Name: disease_risk, dtype: int64

In [36]:
y.shape

(10000,)

In [37]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

num_features = X.select_dtypes(include=['int64', 'float64']).columns
cat_features = ['gender']   # only one categorical column

preprocessor = ColumnTransformer(
    transformers=[
        ('gender_enc', OneHotEncoder(drop='first'), cat_features),
        ('num_scaler', StandardScaler(), num_features)
    ]
)


In [38]:
X = preprocessor.fit_transform(X)

In [39]:
X.shape

(10000, 14)

In [40]:
#sperating dataset into Traning and Testing 
from sklearn.model_selection import train_test_split
X_train , X_test ,y_train , y_test = train_test_split(X , y, test_size = 0.2,random_state = 42)
X_train.shape,X_test.shape

((8000, 14), (2000, 14))

Creating an Evaluate Function to give all metrics after model Traning 

In [41]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, mse, rmse, r2_square

In [42]:
import xgboost
import catboost


In [43]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(), 
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}
model_list =[]
MSE_list=[]

for model_name, model in models.items():

    print(f"Training {model_name}...")

    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae, model_train_mse, model_train_rmse, model_train_r2 = evaluate_model(
        y_train, y_train_pred
    )
    model_test_mae, model_test_mse, model_test_rmse, model_test_r2 = evaluate_model(
        y_test, y_test_pred
    )

    print(model_name)
    print("Train MSE:", round(model_train_mse, 4))
    print("Test  MSE:", round(model_test_mse, 4))
    print("="*40)

    model_list.append(model_name)
    MSE_list.append(model_test_mse)


Training Linear Regression...
Linear Regression
Train MSE: 0.1841
Test  MSE: 0.1881
Training Lasso...
Lasso
Train MSE: 0.1843
Test  MSE: 0.1875
Training Ridge...
Ridge
Train MSE: 0.1841
Test  MSE: 0.1881
Training K-Neighbors Regressor...
K-Neighbors Regressor
Train MSE: 0.1464
Test  MSE: 0.2227
Training Decision Tree...
Decision Tree
Train MSE: 0.0
Test  MSE: 0.3925
Training Random Forest Regressor...
Random Forest Regressor
Train MSE: 0.0267
Test  MSE: 0.1936
Training XGBRegressor...
XGBRegressor
Train MSE: 0.0412
Test  MSE: 0.2209
Training CatBoosting Regressor...
CatBoosting Regressor
Train MSE: 0.0943
Test  MSE: 0.1963
Training AdaBoost Regressor...
AdaBoost Regressor
Train MSE: 0.1844
Test  MSE: 0.1894


In [46]:
pd.DataFrame(list(zip(model_list,MSE_list)),columns = ["Model Name","MSE"]).sort_values(by=["MSE"],ascending=False)

,Model Name,MSE
4,Decision Tree,0.392500
3,K-Neighbors Regressor,0.222700
6,XGBRegressor,0.220935
7,CatBoosting Regressor,0.196253
5,Random Forest Regressor,0.193556
8,AdaBoost Regressor,0.189432
0,Linear Regression,0.188080
2,Ridge,0.188079
1,Lasso,0.187539


In [50]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Create model object
tree_model = DecisionTreeRegressor(random_state=42)

# 2. Train model
tree_model.fit(X_train, y_train)

# 3. Predict
y_pred = tree_model.predict(X_test)

# 4. Evaluate
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.4f}")
print(f"R2 Score: {r2:.4f}")


Mean Squared Error: 0.3905
R2 Score: -1.0827
